# Evaluating Retrieval Quality with Context Recall

Welcome to the evaluation phase of our advanced RAG journey. While building a robust retrieval-augmented generation (RAG) pipeline is complex, ensuring its *reliability* is even more critical. This notebook introduces **Context Recall**, a fundamental metric for assessing whether your retriever component successfully gathered all the necessary information required to answer a given query.

In simple terms, Context Recall answers the question: "Did the retrieved context provide everything needed?" It compares the ground truth (the ideal, complete answer) against the actual retrieved documents. If the source material is missing key facts or concepts—even if it provides some relevant details—the Context Recall score will be low. For advanced systems built with frameworks like LangGraph, knowing this metric allows you to pinpoint exactly where your pipeline fails: Is the LLM failing (generation issue), or did the retriever fail (retrieval issue)?

Mastering metrics like Context Recall is essential for moving beyond simple prototyping and building production-grade AI applications. By quantifying how complete your context is, you can systematically debug and improve every stage of your RAG pipeline—from chunking strategies to embedding models and vector store indexing—ensuring that the LLM always has a comprehensive foundation upon which to build its answer.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Define Context Recall:** Explain what Context Recall measures in the context of RAG evaluation (i.e., the completeness of retrieved information).
*   **Implement Evaluation Metrics:** Utilize the `ragas` library to programmatically calculate the Context Recall score for a given set of inputs, contexts, and ground truths.
*   **Diagnose Retrieval Failures:** Analyze low Context Recall scores to diagnose whether your RAG system is suffering from insufficient context retrieval rather than poor generation capabilities.
*   **Improve Pipeline Robustness:** Understand how improving the retriever component (e.g., better chunking or indexing) directly impacts the reliability and accuracy of the final generated answer.


### Context Recall Scorer Initialization

This cell initializes the necessary components to calculate context recall. It sets up an asynchronous OpenAI client, uses `llm_factory` to get a specific LLM instance (here, 'gpt-5-mini'), and finally instantiates the `ContextRecall` class from Ragas, which requires the configured LLM for scoring.


In [6]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRecall

# Initialize the asynchronous OpenAI client to interact with the API.
client = AsyncOpenAI()

# Use llm_factory to create a specific LLM instance (e.g., 'gpt-5-mini') using the initialized client.
llm = llm_factory("gpt-5-mini", client=client)

# Instantiate the ContextRecall scorer, passing the configured LLM object which is required for metric calculation.
scorer = ContextRecall(llm=llm)


### Context Recall Scoring

This cell demonstrates the 'Context Recall' scoring mechanism. It uses an asynchronous scorer (`scorer.ascore`) to evaluate how well a set of retrieved contexts covers all necessary information (symptoms and causes) when compared against a comprehensive reference answer, highlighting gaps in retrieval.


In [8]:
# Example 1: Contexts cover symptoms but miss the cause (insulin resistance / obesity)
result = await scorer.ascore(
    user_input="What are the symptoms and causes of Type 2 diabetes?",
    retrieved_contexts=[
        "Type 2 diabetes symptoms include frequent urination and excessive thirst.",
        "People with Type 2 diabetes often experience fatigue and blurred vision."
    ],
    reference="Type 2 diabetes is caused by insulin resistance, often linked to obesity and a sedentary lifestyle. Its symptoms include frequent urination, excessive thirst, fatigue, blurred vision, and slow-healing sores."
)
print(f"Context Recall Score: {result.value}")


Context Recall Score: 0.5714285714285714


### Context Recall Scoring (Perfect Match)

This cell demonstrates the calculation of the Context Recall score when the retrieved context perfectly covers all claims made in the ground truth reference. The `scorer.ascore` method is used to quantify how well the provided context supports the user's query and the expected answer.


In [3]:
# Example 2 : The retrieved context fully covers every claim in the ground truth
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",  # The original user question.
    retrieved_contexts=[
        "The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France."
    ],
    reference="The Eiffel Tower is located in Paris, France."
)
print(f"Context Recall Score: {result.value}") # Print the calculated Context Recall score.


Context Recall Score: 1.0


### Context Recall Scoring

This cell demonstrates the 'Context Recall' scoring mechanism, which evaluates how much of the comprehensive `reference` information (the ground truth) is covered by the limited `retrieved_contexts`. It is crucial for assessing if a simple retrieval step provided enough context to fully answer the user's query.


In [4]:
# Example 3: Context only addresses treatment; ground truth covers causes and symptoms
result = await scorer.ascore(
    user_input="What causes and characterizes Parkinson's disease?", # The original question asked by the user.
    retrieved_contexts=[
        "Parkinson's disease is managed using medications such as levodopa and physical therapy to improve quality of life." # Context only mentions treatment/management.
    ],
    reference="Parkinson's disease is caused by the loss of dopamine-producing neurons in the brain. It is characterized by tremors, stiffness, slowness of movement, and balance problems."
)
print(f"Context Recall Score: {result.value}")


Context Recall Score: 0.0
